# Introducción

En esta practica se implementa un pipeline completo de Machine Learning para predecir la probabilidad de impago de préstamos personales usando datos históricos de LendingClub (2007-2017), una plataforma de préstamos peer-to-peer que conecta prestatarios con inversores.

El problema es de clasificación binaria:
- Clase positiva (1 = Default): el prestatario no devuelve el préstamo
- Clase negativa (0 = Fully Paid): el préstamo es devuelto satisfactoriamente

Las clases están desbalanceadas (~80% Fully Paid vs ~20% Default), lo que
condiciona tanto las técnicas de modelado como la elección de métricas de evaluación.

El pipeline se estructura en cuatro etapas: Preprocesamiento, filtrado, modelado y evaluación.


# Preprocesamiento de datos

La clase `Practica1Preprocess` sigue el mismo patrón `fit/transform` que `BasePreprocess`, garantizando que ningún parámetro se aprenda con datos de test (sin data leakage). Las principales diferencias respecto a la clase base son:

### Variables

Se usa el archivo `variables_withExperts.xlsx`, que incluye todas las variables y la de expertos. Por ejemplo grade, sub_grade, relacionadas a fico, entre otras.


### Imputación de missings
  - Numéricas: Se usa `SimpleImputer` con estrategia de mediana. La mediana es robusta a outliers, frecuente en datos financieros como ingresos extremos, balances elevados.
  - Categóricas: Al igual que en las numéricas, se usa `SimpleImputer` pero con estrategia constante para reemplazar los nulos con la categoría `DESCONOCIDO`.  Esto permite que los encoders posteriores aprendan el comportamiento específico de los valores ausentes. Imputar con la moda lo ocultaría mezclándolo con la categoría más frecuente.

### Procesamiento de variables categóricas
  
Se distinguen 4 tipos de variables:

- Ordinales: Las variables `grade`, `subgrade`, `emp_length` tienen orden natural con significado crediticio. Por ejemplo, en `grade` se tiene A=menor riesgo y G=mayor riesgo. Para este tipo de variables se usa `OrdinalEncoder`.
- Binarias: Las variables `term`, `application_type` poseen solo dos valores. Por lo tanto, se realiza un mapeo directo a 0 y 1.
- Texto: Las variables `emp_title` y `desc` poseen texto de longitud variable. Se mantienen opcionales debido a su alto coste computacional y a que, en los experimentos realizados, no aportan mejoras significativas bajo el enfoque actual.
- Nominales restantes: El resto de variables se condifican con `TargetEncoder`  que sustituye cada categoría por una estimación de la media del target. Este enfoque reduce la dimensionalidad respecto a OneHotEncoder.

### Procesamiento de variables numéricas

Se usa `RobustScaler`, el cual estandariza cada variable usando la mediana como centro y el IQR como escala. Esto lo hace robusto a outliers.

### Generación de nuevas features

En lugar de `PolynomialFeatures` se construyen ratios con significado financiero directo. Las variables generadas son:

- Promedio de los límites de rango FICO: Esta variable reduce la redundancia y captura el score central (media).
- Ratio de cuota mensual sobre ingreso mensual: Mide la carga financiera real del prestatario.
- Ratio de utilización de crédito de revolving: Un alto uso del crédito disponible puede indicar riesgo financiero.
- Importe del préstamo sobre el ingreso anual: Captura si la deuda es proporcional a la capacidad de pago.
- Ingreso anual discretizado en 5 grupos: Permite capturar relaciones no lineales entre ingresos y riesgo que puede ser de utilidad en algunos modelos lineales.
- Antiguedad del historial crediticio: Clientes con mayor antiguedad suelen tener un comportamiento financiero más estable.





### Parametros usados para el preprocesamiento

- `use_text_vars=False`: no se incluyen las variables de texto largo debido a su alto coste computacional para obtener vectores característicos. Además, en los experimentos realizados estas variables no han sido seleccionadas por el proceso de filtrado, lo que indica que no aportan información relevante para este problema.

- `random_state=42`: garantiza la reproducibilidad del proceso de encoding basado en target encoding, el cual utiliza validación cruzada interna.

In [6]:
from src.preprocessing.practica1_preprocessing import Practica1Preprocess

# Instanciamos la clase de preprocesamiento.
# El fichero Excel contiene la lista de variables candidatas a ser predictoras.
practica_pre = Practica1Preprocess("data/variables_withExperts.xlsx", "loan_status", use_text_vars=False, random_state=42)

In [7]:
# fit(): aprende los parametros del preprocesamiento SOLO con datos de entrenamiento.
# Esto incluye: categorias del OHE, medianas para imputacion, parametros del QuantileTransformer, etc.
practica_pre.fit("data/df_train_small.csv")

/src/preprocessing/practica1_preprocessing.py:63: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  self.train_X_data['earliest_cr_line'] = pd.to_datetime(self.train_X_data['earliest_cr_line'])


In [8]:
# transform(): aplica las transformaciones aprendidas en fit().
# Devuelve X_train (features) e y_train (target: True=default, False=fully paid).
X_train, y_train = practica_pre.transform("data/df_train_small.csv")
print(f"Dimensiones tras preprocesamiento: {X_train.shape[0]} filas x {X_train.shape[1]} columnas")

/src/preprocessing/practica1_preprocessing.py:191: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  X_data['earliest_cr_line'] = pd.to_datetime(X_data['earliest_cr_line'])


Dimensiones tras preprocesamiento: 80000 filas x 91 columnas


# Filtrado de features

La clase `Practica1Filtering` sigue el mismo patrón `fit/transform` que `BaseFiltering`, garantizando que ningún parámetro se aprenda con datos de test (sin data leakage). Las principales diferencias respecto a la clase base son:

### Eliminación por varianza


Se usa `VarianceThreshold` para eliminar las variables cuya varianza es menor a un umbral (por defecto 0.01), es decir, variables casi constantes o con muy poca variabilidad. Una variable con con varianza baja aporta poca información al modelo y puede introducir ruido. Es una alternativa más general basada en la varianza, mientras que enfoques como `DropConstantFeatures` detectan variables constantes o casi constantes en función de la frecuencia del valor más repetido.

Es posible definir el umbral mediante el parámetro ``variance_threshold`` en la clase ``Practical1Filtering``.

### Selección univariada

Se seleccionan las *k* variables más relevantes respecto al target usando información mutua (`mutual_info_classif`) con `SelectKBest`. La información mutua mide la dependencia estadística entre cada feature y el target de forma no lineal, a diferencia de la correlación de Pearson que solo detecta relaciones lineales. Esto es importante en datos crediticios, donde muchas relaciones relevantes (por ejemplo, entre nivel de endeudamiento y probabilidad de impago) no son estrictamente lineales.

Es posible definir el valor de *k* mediante el parámetro ``k_best`` en la clase ``Practical1Filtering``.


### Selección basada en modelo Random Forest

Se usa `SelectFromModel` con Random Forest para estimar la importancia de cada variable y eliminar aquellas cuya importancia este por debajo de un umbral (por defecto, la media de las importancias). Se usa `class_weight=balanced` para que el modelo no ignore la clase minoritaria al calcular las importancias.

Es posible definir el valor del umbral, número de árboles y profundidad máxima mediante los parámetros ``rf_threshold``, ``rf_n_estimators``, ``rf_max_depth`` respectivamente.



### Parametros usados para el filtrado

- `variance_threshold=0.01`: se utiliza un umbral bajo para eliminar únicamente variables casi constantes. Esto permite conservar la mayoría de la información útil del dataset, eliminando solo features con baja variabilidad que no aportan poder predictivo.

- `k_best=30`: se selecciona un número intermedio de variables relevantes según información mutua. Con este valor se reduce la dimensionalidad inicial (91 features) sin eliminar demasiada información potencialmente útil.

- `rf_n_estimators=200`: un número elevado de árboles en el Random Forest reduce la varianza del modelo y hace más estables las estimaciones de importancia de las variables.

- `rf_max_depth=None`: se permite que los árboles crezcan completamente para capturar interacciones complejas entre variables. Esto es útil en seleccion de variables, donde el objetivo es medir importancia real y no regularizar el modelo final.

- `rf_threshold="mean"`: se utiliza un criterio relativo que elimina variables con importancia inferior al promedio, manteniendo las demas.

- `random_state=42`: garantiza la reproducibilidad del proceso de selección de variables.

In [9]:
from src.filtering.practica1_filtering import Practica1Filtering

# Instanciamos el filtro con los parametros por defecto.
# Todos los parametros son configurables en el constructor.
practica_filter = Practica1Filtering(
    variance_threshold=0.01,
    k_best=30,
    rf_threshold="mean",
    rf_n_estimators=200,
    rf_max_depth=None,
    random_state=42
)

In [10]:
# fit(): aprende que features eliminar usando SOLO datos de train.
# Internamente ejecuta los 3 filtros en secuencia.
practica_filter.fit(X_train, y_train)

In [11]:
# Resumen del filtrado: cuantas features se eliminaron en cada paso.
practica_filter.print_summary()

RESUMEN DEL PIPELINE DE FILTRADO
  Features iniciales:              91
  Eliminadas por baja varianza:     -9
  Eliminadas por SelectkBest:       -52
  Eliminadas por importancia de RF:      -17
  Features seleccionadas finales:  13


In [12]:
# transform(): aplica los filtros aprendidos en fit() a los datos de train.
X_train_filtered = practica_filter.transform(X_train)

print(f"Features seleccionadas ({X_train_filtered.shape[1]}):")
print(X_train_filtered.columns.tolist())
print(f"Dimensiones tras filtrado: {X_train_filtered.shape[0]} filas x {X_train_filtered.shape[1]} columnas")

Features seleccionadas (13):
['loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti', 'tot_cur_bal', 'avg_cur_bal', 'bc_open_to_buy', 'mo_sin_old_il_acct', 'installment_income_ratio', 'debt_income_ratio', 'credit_age', 'sub_grade']
Dimensiones tras filtrado: 80000 filas x 13 columnas


Tras el proceso de selección de variables, el conjunto final queda reducido a 13 variables. Estas variables combinan información sobre capacidad de pago, nivel de endeudamiento y utilización de crédito, y perfil crediticio e histórico.

# Data de prueba

La data de prueba se procesa utilizando las instancias previamente entrenadas en el conjunto de entrenamiento, aplicando únicamente el método `transform` tanto en el preprocesamiento como en el filtrado de variables. Esto asegura que no exista*data leakage, ya que ningún parámetro es recalculado con información del test.

In [13]:
# Preprocesamiento del test (transform, NO fit)
X_test, y_test = practica_pre.transform("data/df_test_small.csv")

# Filtrado del test (transform, NO fit)
X_test_filtered = practica_filter.transform(X_test)

print(f"Dimensiones train filtrado: {X_train_filtered.shape}")
print(f"Dimensiones test filtrado:  {X_test_filtered.shape}")

/src/preprocessing/practica1_preprocessing.py:191: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  X_data['earliest_cr_line'] = pd.to_datetime(X_data['earliest_cr_line'])


Dimensiones train filtrado: (80000, 13)
Dimensiones test filtrado:  (20000, 13)


Las etiquetas del conjunto de prueba se convierten a un arreglo unidimensional para facilitar el cálculo de métricas de evaluación del modelo.

In [14]:
y_test_flat = y_test.values.ravel()

# Modelo base

Se utiliza el modelo base proporcionado. Para ello se replica su implementación con pequeños cambios en el etiquetado para compararlo correctamente con los modelos que se entrenarán más adelante.

In [15]:
import pandas as pd
import numpy as np

#Leemos el dataset
test_df = pd.read_csv('data/df_test_small.csv')

In [16]:
[n for n in test_df.columns if 'fico' in n]

['fico_range_low',
 'fico_range_high',
 'last_fico_range_high',
 'last_fico_range_low',
 'sec_app_fico_range_low',
 'sec_app_fico_range_high']

In [17]:
# nos quedamos con puntajes fico
df_fico = ( test_df
        .loc[:,["fico_range_low", "fico_range_high", "loan_status"]]
        .assign(prob_low = lambda x: (x.fico_range_low - 300) / (850 - 300))
        .assign(prob_high = lambda x: (x.fico_range_high - 300) / (850 - 300))
        .assign(prob = lambda x: (x.prob_low + x.prob_high) / 2)
        .assign(loan_paid = lambda x: np.where(x.loan_status != "Fully Paid", 1, 0))
)

In [18]:
# El límite estándar de FICO es 670, por lo que podemos usarlo como umbral para predecir si un préstamo se pagará o no.
df_fico = df_fico.assign(prediction = lambda x: np.where(x.prob > 0.67, 0, 1))

## Predicciones del modelo base

Se obtienen las predicciones del modelo base

In [19]:
class_predicted_base = df_fico.prediction
prob_predicted_base = df_fico.prob

## Métricas de evaluación

In [20]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)
import matplotlib.pyplot as plt
import numpy as np

# Classification Report
print("=" * 60)
print("REPORTE DE CLASIFICACIÓN DEL MODELO BASE")
print("=" * 60)
print(classification_report(y_test_flat, class_predicted_base,
                            target_names=["Fully Paid (False)", "Default (True)"]))

REPORTE DE CLASIFICACIÓN DEL MODELO BASE
                    precision    recall  f1-score   support

Fully Paid (False)       0.81      0.84      0.83     16003
    Default (True)       0.26      0.24      0.25      3997

          accuracy                           0.72     20000
         macro avg       0.54      0.54      0.54     20000
      weighted avg       0.70      0.72      0.71     20000



# Entrenamiento de modelos

Se entrenan tres modelos, uno de cada familia requerido:

- `RandomForestClassifier`: Ensamble de árboles de decisión
- `SVC con kernel rbf`: Máquina de soporte vectorial
- `MLPClassifier`: Red neuronal multicapa

Para abordar el desbalance de clases, se emplean estrategias de ponderación como `class_weight="balanced"` y `sample_weight`, que ajustan el aprendizaje para dar mayor relevancia a la clase minoritaria.

## Random Forest

### Entrenamiento del modelo

Se utiliza un `RandomForestClassifier` como modelo de ensamble, el cual combina múltiples árboles de decisión para reducir la varianza y mejorar la generalización.

Parametros especificos de Random Forest:

- `n_estimators=100`: número de árboles suficiente para estabilizar el rendimiento del modelo sin un coste computacional excesivo.
- `max_depth=10`: limita la profundidad de los árboles para reducir el riesgo de sobreajuste, forzando al modelo a capturar patrones más generales.
- `class_weight="balanced"`: ajusta automáticamente los pesos de las clases inversamente a su frecuencia para dar mayor importancia a la clase minoritaria.
- `random_state=42`: para reproducibilidad de los resultados.

Los parametros se seleccionaron mediante distintos experimentos, variando principalmente `max_depth` y `n_estimators`. Los valores elegidos corresponden a la configuración que mostró el mejor desempeño en las pruebas realizadas.


In [21]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced',
    max_depth = 10
)
rf_model.fit(X_train_filtered, y_train.values.ravel())

train_accuracy_rf = rf_model.score(X_train_filtered, y_train.values.ravel())
print(f"Modelo entrenado. Accuracy en TRAIN: {train_accuracy_rf:.4f}")

Modelo entrenado. Accuracy en TRAIN: 0.6970


### Predicciones del modelo

In [22]:
# Predicciones del modelo
# predict(): devuelve la clase predicha (True/False)
# predict_proba()[:,1]: devuelve la probabilidad de la clase positiva (Default)
class_predicted_rf = rf_model.predict(X_test_filtered)
prob_predicted_rf = rf_model.predict_proba(X_test_filtered)[:, 1]

### Métricas de evaluación

In [23]:
# Accuracy
test_accuracy = accuracy_score(y_test_flat, class_predicted_rf)
print(f"Accuracy en TEST:  {test_accuracy:.4f}")
print(f"Accuracy en TRAIN: {train_accuracy_rf:.4f}")
print(f"Diferencia (posible overfitting): {train_accuracy_rf - test_accuracy:.4f}")

Accuracy en TEST:  0.6540
Accuracy en TRAIN: 0.6970
Diferencia (posible overfitting): 0.0431


#### Reporte de clasificación

In [24]:
# Classification Report
print("=" * 60)
print("REPORTE DE CLASIFICACIÓN DE RANDOM FOREST")
print("=" * 60)
print(classification_report(y_test_flat, class_predicted_rf,
                            target_names=["Fully Paid (False)", "Default (True)"]))

REPORTE DE CLASIFICACIÓN DE RANDOM FOREST
                    precision    recall  f1-score   support

Fully Paid (False)       0.88      0.66      0.75     16003
    Default (True)       0.32      0.63      0.42      3997

          accuracy                           0.65     20000
         macro avg       0.60      0.64      0.59     20000
      weighted avg       0.76      0.65      0.69     20000



## SVM con kernel RBF

### Muestreo estratificado

SVM es demasiado lento con muchas datos como el dataset actual (80000 datos), por lo que se realiza un muestreo estratificado de 10000 muestras para reducir el coste computacional manteniendo la proporción original de clases.

El muestreo se realiza con `StratifiedShuffleSplit`, que garantiza que la distribución de la variable objetivo se preserve en el subconjunto utilizado para entrenamiento del modelo.


In [25]:
from sklearn.model_selection import StratifiedShuffleSplit
# Número máximo de muestras a utilizar para entrenar el modelo SVM
# (se usa una submuestra para reducir costo computacional)
MAX_SAMPLES = 10000

# StratifiedShuffleSplit asegura que la distribución de clases
# se mantenga proporcional en la submuestra respecto al dataset original
stratified_split = StratifiedShuffleSplit(n_splits=1, train_size=MAX_SAMPLES, random_state=42)
train_index, _ = next(stratified_split.split(X_train_filtered, y_train.values.ravel()))

# Subconjunto de features y etiquetas
X_train_svm = X_train_filtered.iloc[train_index]
y_train_svm = y_train[train_index].values.ravel()

### Entrenamiento del modelo

Se utiliza un `SVC` con kernel RBF, el cual permite modelar relaciones no lineales mediante la proyección implícita de los datos a un espacio de mayor dimensión.

Parametros especificos de SVM:

- `kernel="rbf"`: permite capturar relaciones no lineales complejas entre variables, lo cual es adecuado en problemas de riesgo crediticio donde las fronteras de decisión no son lineales.
- `C=20.0`: controla el trade-off entre margen de separación y error de clasificación. Un valor alto penaliza más los errores. Sirve para regularizar y reducir sobreajuste.
- `class_weight="balanced"`: ajusta los pesos de las clases para compensar el desbalance, dando mayor importancia a la clase minoritaria.
- `probability=True`: habilita la estimación de probabilidades, necesaria para análisis posteriores y comparación con otros modelos.
- `random_state=42`: asegura la reproducibilidad del muestreo y del entrenamiento.

Los parametros se seleccionaron mediante distintos experimentos, ajustando principalmente el parametro C. Valores bajos de este parámetro provocaban un bajo recall y precisión en la clase positiva, mientras que al incrementarlo se observó una mejora en el desempeño del modelo para dicha clase.


In [26]:
from sklearn.svm import SVC

svm_model = SVC(
    kernel="rbf",
    C=20.0,
    probability=True,
    class_weight="balanced",
    random_state=42
)

svm_model.fit(X_train_svm, y_train_svm)

train_accuracy_svm = svm_model.score(X_train_svm, y_train_svm)
print(f"SVM entrenado. Accuracy en TRAIN: {train_accuracy_svm:.4f}")

SVM entrenado. Accuracy en TRAIN: 0.6852


### Predicciones del modelo

In [27]:
class_predicted_svm = svm_model.predict(X_test_filtered)
prob_predicted_svm = svm_model.predict_proba(X_test_filtered)[:, 1]

### Métricas de evaluación

In [28]:
# Accuracy
test_accuracy_svm = accuracy_score(y_test_flat, class_predicted_svm)
print(f"Accuracy en TEST:  {test_accuracy_svm:.4f}")
print(f"Accuracy en TRAIN: {train_accuracy_svm:.4f}")
print(f"Diferencia (posible overfitting): {train_accuracy_svm - test_accuracy_svm:.4f}")

Accuracy en TEST:  0.6805
Accuracy en TRAIN: 0.6852
Diferencia (posible overfitting): 0.0047


#### Reporte de clasificación

In [29]:
# Classification Report
print("=" * 60)
print("REPORTE DE CLASIFICACIÓN DE SVM")
print("=" * 60)
print(classification_report(y_test_flat, class_predicted_svm,
                            target_names=["Fully Paid (False)", "Default (True)"]))

REPORTE DE CLASIFICACIÓN DE SVM
                    precision    recall  f1-score   support

Fully Paid (False)       0.86      0.72      0.78     16003
    Default (True)       0.32      0.54      0.40      3997

          accuracy                           0.68     20000
         macro avg       0.59      0.63      0.59     20000
      weighted avg       0.75      0.68      0.71     20000



## Red neuronal

### Entrenamiento del modelo

Se utiliza un `MLPClassifier`, una red neuronal multicapa, capaz de modelar relaciones no lineales complejas entre las variables de entrada mediante capas densas.

Parametros especificos de la red neuronal:

- `hidden_layer_sizes=(100, 100)`: define una arquitectura con dos capas ocultas de 100 neuronas cada una, lo que permite al modelo aprender representaciones no lineales más complejas sin ser demasiado profundo.
- `activation="relu"`: función de activación no lineal que mejora la eficiencia del entrenamiento.
- `solver="adam"`: optimizador adecuado para datasets grandes.
- `alpha=0.0001`: término de regularización L2 que ayuda a controlar el sobreajuste penalizando pesos grandes.
- `learning_rate_init=0.0001`: tasa de aprendizaje inicial baja para estabilizar el entrenamiento y evitar oscilaciones en la convergencia.
- `batch_size=32`: entrenamiento en mini-batches para mejorar la eficiencia computacional. Valores pequeños dieron mejores resultados.
- `early_stopping=True`: detiene el entrenamiento si el rendimiento en validación no mejora, actuando como mecanismo adicional de regularización.
- `validation_fraction=0.2`: proporción de datos de entrenamiento utilizada para validación interna durante early stopping.
- `max_iter=100`: límite superior de iteraciones para evitar entrenamientos excesivamente largos.
- `random_state=42`: garantiza la reproducibilidad del proceso de entrenamiento.

A diferencia de los modelos anteriores, en este modelo no se tiene le parametro ``class_weight``. Por ello, se utiliza la función `compute_sample_weight("balanced")` que permite obtener pesos inversamente proporcionales a la frecuencia de cada clase y que pueden ser usados en el método fit.

Los parametros se seleccionaron mediante distintos experimentos. Se exploraron principalmente configuraciones de `hidden_layer_sizes`, variando el número de capas y neuronas. En el caso de `learning_rate_init`, se observó que valores altos impedían la convergencia del modelo, por lo que se redujo su magnitud. Para `batch_size`, se utilizaron tamaños pequeños, ya que incrementos mayores no mostraron mejoras en el desempeño. Finalmente, el número máximo de épocas se definió en función del `early_stopping`, el cual detuvo el entrenamiento en menos de 100 iteraciones en la mayoría de los casos.



In [30]:
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight

mlp_model = MLPClassifier(
    hidden_layer_sizes=(100, 100),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    learning_rate_init=0.0001,
    random_state=42,
    early_stopping=True,
    max_iter=100,
    batch_size=32,
    validation_fraction=0.2
)

sample_weight_train = compute_sample_weight("balanced", y_train.values.ravel())
mlp_model.fit(X_train_filtered, y_train.values.ravel(), sample_weight=sample_weight_train)

train_accuracy_mlp = mlp_model.score(X_train_filtered, y_train.values.ravel())
print(f"MLP entrenado. Accuracy en TRAIN: {train_accuracy_mlp:.4f}")
print(f"Epocas de entrenamiento: {mlp_model.n_iter_}")

MLP entrenado. Accuracy en TRAIN: 0.6682
Epocas de entrenamiento: 35


### Predicciones del modelo

In [31]:
class_predicted_mlp = mlp_model.predict(X_test_filtered)
prob_predicted_mlp = mlp_model.predict_proba(X_test_filtered)[:, 1]

### Métricas de evaluación

In [32]:
# Accuracy
test_accuracy_mlp = accuracy_score(y_test_flat, class_predicted_mlp)
print(f"Accuracy en TEST:  {test_accuracy_mlp:.4f}")
print(f"Accuracy en TRAIN: {train_accuracy_mlp:.4f}")
print(f"Diferencia (posible overfitting): {train_accuracy_mlp - test_accuracy_mlp:.4f}")

Accuracy en TEST:  0.6634
Accuracy en TRAIN: 0.6682
Diferencia (posible overfitting): 0.0048


#### Reporte de clasificación

In [33]:
# Classification Report
print("=" * 60)
print("REPORTE DE CLASIFICACIÓN DE RED NEURONAL")
print("=" * 60)
print(classification_report(y_test_flat, class_predicted_mlp,
                            target_names=["Fully Paid (False)", "Default (True)"]))

REPORTE DE CLASIFICACIÓN DE RED NEURONAL
                    precision    recall  f1-score   support

Fully Paid (False)       0.88      0.67      0.76     16003
    Default (True)       0.32      0.62      0.43      3997

          accuracy                           0.66     20000
         macro avg       0.60      0.65      0.59     20000
      weighted avg       0.77      0.66      0.69     20000



# Comparación de modelos

Para evaluar el desempeño de los distintos modelos, se define una función de cálculo de métricas estándar de clasificación junto con una métrica adicional más adecuada para problemas desbalanceados: el área bajo la curva Precision-Recall (PR-AUC).

El uso de PR-AUC es especialmente relevante en este contexto, ya que permite evaluar la capacidad del modelo para identificar la clase minoritaria sin verse afectado por el desbalance de clases, a diferencia de métricas como accuracy.


In [34]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, auc
)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def calcular_metricas(nombre_modelo, y_true, y_pred, y_prob):
    """
    Calcula métricas de evaluación para un modelo de clasificación binaria.
    Retorna un diccionario con métricas calculadas del modelo actual
    """
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = auc(recall_vals, precision_vals)

    return {
        "Modelo"   : nombre_modelo,
        "Accuracy" : acc,
        "Precision": precision,
        "Recall"   : recall,
        "F1-score" : f1,
        "PR-AUC"   : pr_auc
    }


def generar_tabla_resultados(y_true, predicciones):
    """
    Genera una tabla comparativa de métricas para múltiples modelos.
    Retorna un dataFrame ordenado por PR-AUC de mayor a menor
    """
    lista_resultados = []
    for nombre_modelo, (y_pred, y_prob) in predicciones.items():
        lista_resultados.append(calcular_metricas(nombre_modelo, y_true, y_pred, y_prob))
    return pd.DataFrame(lista_resultados).sort_values("PR-AUC", ascending=False).round(4)


In [35]:
predicciones = {
    "Modelo Base"  : (class_predicted_base, prob_predicted_base),
    "Random Forest": (class_predicted_rf, prob_predicted_rf),
    "SVM"          : (class_predicted_svm, prob_predicted_svm),
    "Red Neuronal" : (class_predicted_mlp, prob_predicted_mlp)
}

print("=" * 80)
print("COMPARACION DE MODELOS (ordenados por PR-AUC)")
print("=" * 80)
tabla_resultados = generar_tabla_resultados(y_test_flat, predicciones)
tabla_resultados

COMPARACION DE MODELOS (ordenados por PR-AUC)


,Modelo,Accuracy,Precision,Recall,F1-score,PR-AUC
3,Red Neuronal,0.6634,0.3229,0.6237,0.4255,0.3653
1,Random Forest,0.6540,0.3160,0.6282,0.4205,0.3613
2,SVM,0.6805,0.3218,0.5407,0.4035,0.3442
0,Modelo Base,0.7166,0.2647,0.2352,0.2491,0.1592


## Interpretación de Resultados

Los tres modelos superan al modelo base en Recall y PR-AUC, que son las métricas
más relevantes en detección de impago. Un accuracy alto no es suficiente con clases desbalanceadas, ya que un modelo trivial que prediga siempre "no impago" puede obtener un accuracy elevado sin ser útil. El modelo base alcanza un accuracy de 0.717 precisamente por este motivo, pero su recall en la clase de impago es muy bajo (0.235), lo que implica que no detecta la mayoría de los clientes en riesgo, lo cual lo hace inadecuado para un entorno real de crédito.

PR-AUC es la métrica principal de comparación porque, a diferencia de ROC-AUC,
se centra exclusivamente en el rendimiento sobre la clase minoritaria (impago).
Recall es la métrica más crítica en este problema. Un impago no detectado implica pérdida directa del capital prestado, mientras que rechazar un buen cliente solo implica pérdida de negocio.


### ¿Qué modelo funciona mejor?

La Red Neuronal y Random Forest son los modelos más adecuados para este problema. Ambos alcanzan un recall superior a 0.62 y un PR-AUC en torno a 0.36, mostrando una mejora significativa respecto al modelo base. La diferencia entre ambos es mínima. Random Forest tiene la ventaja adicional de ser más interpretable, lo cual es un requisito habitual en entornos bancarios.

SVM presenta un rendimiento intermedio, tiene un accuracy más alto (0.681) pero el recall más bajo de los tres (0.541), lo que indica una mayor tendencia a favorecer la clase mayoritaria en el proceso de separación, posiblemente debido al muestreo utilizado y los parametros elegidos.

Los resultados podrían mejorarse mediante una optimización más exhaustiva de parametros y, especialmente, mediante la calibración del umbral de decisión en lugar de utilizar el valor por defecto como 0.5.


### Implicaciones en el contexto bancario

Una precisión de ~0.32 en los tres modelos significa que aproximadamente 1 de cada 3 préstamos marcados como impago resultaría ser un buen cliente rechazado.
Esto se traduce en un coste asociado a falsos positivos, es decir, la posible pérdida de clientes que si cumplen pero fueron rechazados.

Por otro lado, el coste de un falso negativo (conceder un préstamo a un cliente que finalmente incurre en impago) suele ser significativamente mayor, ya que implica pérdidas directas de capital. Por esta razón, en muchos escenarios el objetivo principal es maximizar la detección de impagos (alto recall), incluso a costa de aceptar un mayor número de falsos positivos.

La decisión no se basa en un umbral fijo de 0.5, sino que este se ajusta según la estrategia de la entidad. Reducir el umbral permite identificar más casos de posible impago (mayor recall), aunque también aumenta los falsos positivos. Por otro lado, aumentar el umbral hace que las predicciones sean más precisas, pero reduce la capacidad de detectar clientes con riesgo. Por ello, es necesario ajustar este valor en función del equilibrio deseado entre detectar riesgos y evitar rechazar buenos clientes, idealmente apoyado en un análisis coste-beneficio antes de su implementación en producción.
